# 02.2 Introduction to ETL with PySpark & Minio

In this laboratory we will be learning the basics about using PySpark to implement ETL pipelines with a booking database as an example

![schema](imgs/bookings-schema.png)

## 1. Moving data from local storage to Minio

In this task you will need to move the files located in `data/bookings` of the local storage to the Minio bucket named `test`, all the files should have the prefix `lab2/landing/` using Apache Spark

In [1]:
s3_bucket_name="test"
s3_landing_prefix = "lab2/landing"
local_file_path = "data/bookings"

In [2]:
# Create and configure your spark session here
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkWithS3Files") \
    .master("local[*]") \
    .getOrCreate()

def load_config(spark_context: SparkContext):
    spark_context._jsc.hadoopConfiguration().set("fs.s3a.access.key", os.getenv("AWS_ACCESS_KEY_ID"))
    spark_context._jsc.hadoopConfiguration().set("fs.s3a.secret.key", os.getenv("AWS_SECRET_ACCESS_KEY"))
    spark_context._jsc.hadoopConfiguration().set("fs.s3a.endpoint", os.getenv("AWS_ENDPOINT"))
    spark_context._jsc.hadoopConfiguration().set("fs.s3a.region", os.getenv("AWS_ENDPOINT"))
    spark_context._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "true")
    spark_context._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
    spark_context._jsc.hadoopConfiguration().set("fs.s3a.attempts.maximum", "1")
    spark_context._jsc.hadoopConfiguration().set("fs.s3a.connection.establish.timeout", "5000")
    spark_context._jsc.hadoopConfiguration().set("fs.s3a.connection.timeout", "10000")

load_config(spark.sparkContext)

26/01/10 13:57:09 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
files = ["bookings", "facilities", "members"]
# Spark by default will create a folder and within it will place the result file in chunks (worker node chunks)

for file in files:
    # upload the file to Minio
    s3_spark_file_key = f"{s3_landing_prefix}/{file}"
    s3_spark_file_url = f"s3a://{s3_bucket_name}/{s3_spark_file_key}"
    
    local_spark_df = spark.read.csv(
        f"{local_file_path}/{file}.csv",   
        header=True,         
        inferSchema=True
    )

    
    local_spark_df.write\
    .format('csv')\
    .option("header", "true")\
    .mode("overwrite")\
    .save(s3_spark_file_url)



26/01/10 13:57:23 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/01/10 13:57:26 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/01/10 13:57:26 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/01/10 13:57:29 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/01/10 13:57:29 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/01/10 13:57:31 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/01/10 13:57:31 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


## 2. Transforming the data

In this section you will have to read the files you moved from the local storage to a landing zone in the `test` bucket and apply some transformations to them. 

Then you will save the transformed data 

PySpark provides multiple functions to trasnform the data, a majority of these are provided in the `pyspark.sql.functions` module

### Column & DataFrame Manipulation

| Function   | Purpose                                 | Example                                                        |
| ---------- | --------------------------------------- | -------------------------------------------------------------- |
| `col`      | Reference a column by name              | `df.select(col("age"))`                                        |
| `lit`      | Create a column with a literal value    | `df.withColumn("country", lit("USA"))`                         |
| `alias`    | Rename a column in a select             | `df.select(col("age").alias("user_age"))`                      |
| `when`     | Conditional expressions (like SQL CASE) | `df.select(when(col("age") > 18, "Adult").otherwise("Minor"))` |
| `coalesce` | Return first non-null value             | `df.select(coalesce(col("phone"), lit("N/A")))`                |


### Aggregation & Grouping

| Function        | Purpose                  | Example                                 |
| --------------- | ------------------------ | --------------------------------------- |
| `count`         | Count rows               | `df.groupBy("country").agg(count("*"))` |
| `countDistinct` | Count distinct values    | `df.agg(countDistinct("user_id"))`      |
| `sum`           | Sum of a column          | `df.agg(sum("sales"))`                  |
| `avg` / `mean`  | Average of a column      | `df.agg(avg("salary"))`                 |
| `max` / `min`   | Maximum or minimum value | `df.agg(max("salary"), min("salary"))`  |


### String Functions
| Function         | Purpose                      | Example                                                            |
| ---------------- | ---------------------------- | ------------------------------------------------------------------ |
| `lower`          | Convert to lowercase         | `df.select(lower(col("name")))`                                    |
| `upper`          | Convert to uppercase         | `df.select(upper(col("name")))`                                    |
| `concat`         | Concatenate columns          | `df.select(concat(col("first_name"), lit(" "), col("last_name")))` |
| `concat_ws`      | Concatenate with a separator | `df.select(concat_ws("-", "year", "month", "day"))`                |
| `substring`      | Extract substring            | `df.select(substring(col("phone"), 1, 3))`                         |
| `trim`           | Trim whitespace              | `df.select(trim(col("username")))`                                 |
| `regexp_extract` | Extract regex match          | `df.select(regexp_extract(col("email"), r"@(.+)", 1))`             |
| `regexp_replace` | Replace regex match          | `df.select(regexp_replace(col("phone"), "-", ""))`                 |


### Date & Time Functions

| Function                      | Purpose                                | Example                                                               |
| ----------------------------- | -------------------------------------- | --------------------------------------------------------------------- |
| `current_date`                | Current date                           | `df.select(current_date())`                                           |
| `current_timestamp`           | Current timestamp                      | `df.select(current_timestamp())`                                      |
| `date_add`                    | Add days to date                       | `df.select(date_add(col("start_date"), 7))`                           |
| `date_sub`                    | Subtract days from date                | `df.select(date_sub(col("start_date"), 7))`                           |
| `datediff`                    | Difference between two dates (in days) | `df.select(datediff(col("end_date"), col("start_date")))`             |
| `months_between`              | Difference in months                   | `df.select(months_between(col("end_date"), col("start_date")))`       |
| `year`, `month`, `dayofmonth` | Extract date parts                     | `df.select(year(col("date")), month(col("date")))`                    |
| `to_date`                     | Convert string to date                 | `df.select(to_date(col("date_string"), "yyyy-MM-dd"))`                |
| `to_timestamp`                | Convert string to timestamp            | `df.select(to_timestamp(col("datetime_str"), "yyyy-MM-dd HH:mm:ss"))` |


### Null handling
| Function                           | Purpose                    | Example                               |
| ---------------------------------- | -------------------------- | ------------------------------------- |
| `isnull`                           | Check for null values      | `df.filter(col("email").isNull())`    |
| `isnotnull`                        | Check for non-null values  | `df.filter(col("email").isNotNull())` |
| `na.fill` *(method, not function)* | Replace nulls with a value | `df.na.fill({"email": "unknown"})`    |
| `na.drop` *(method, not function)* | Drop rows with nulls       | `df.na.drop()`                        |


In [4]:
import pyspark.sql.functions as F

transformed_prefix = "lab2/transformed"

### 2.1 Transforming members data

For members we need to transform the original table so:
- Shows the full name of the members
- Remove the `surname` and `firstname` columns
- The `recommendedby` column should show the full name of the person instead of the id, if the person was not recommended show "NOT RECOMMENDED"


In [8]:
# Create a spark dataframe from the members file uploaded in s3
members_df = spark.read.csv(
    f"s3a://{s3_bucket_name}/{s3_landing_prefix}/members",
    header=True,      # Use first row as column names
    inferSchema=True  # Automatically detect data types
)

In [9]:
members_df.show()

+-----+---------+---------+--------------------+-------+--------------+-------------+-------------------+
|memid|  surname|firstname|             address|zipcode|     telephone|recommendedby|           joindate|
+-----+---------+---------+--------------------+-------+--------------+-------------+-------------------+
|    0|    GUEST|    GUEST|               GUEST|      0|(000) 000-0000|         NULL|2012-07-01 00:00:00|
|    1|    Smith|   Darren|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2012-07-02 12:02:05|
|    2|    Smith|    Tracy|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2012-07-02 12:08:23|
|    3|   Rownam|      Tim|23 Highway Way, B...|  23423|(844) 693-0723|         NULL|2012-07-03 09:32:15|
|    4| Joplette|   Janice|20 Crossing Road,...|    234|(833) 942-4710|            1|2012-07-03 10:25:05|
|    5|  Butters|   Gerald|1065 Huntingdon A...|  56754|(844) 078-4130|            1|2012-07-09 10:44:09|
|    6|    Tracy|   Burton|3 Tunisia Drive, ..

In [10]:
# Transform the spark dataframe
members_df = members_df.withColumn("Fullname", F.concat(F.col("firstname"), F.lit(" "), F.col("surname")))

In [11]:
members_df = members_df.select("memid", "Fullname","address", "recommendedby", "joindate","zipcode", "telephone")

In [12]:
members_df = members_df.alias("m1").join(members_df.alias("m2"),F.col("m2.memid")==F.col("m1.recommendedby"),"left_outer")\
.select("m1.memid", "m1.Fullname","m1.address", "m1.recommendedby", "m1.joindate","m1.zipcode", "m1.telephone", F.col("m2.Fullname").alias("recommendedbyname"))


In [13]:
members_df = members_df.withColumn("recommendedbyname", F.coalesce(F.col("recommendedbyname"), F.lit("NOT RECOMMENDED")))

In [14]:
# You can see the results with df.show()
members_df.show()

+-----+---------------+--------------------+-------------+-------------------+-------+--------------+-----------------+
|memid|       Fullname|             address|recommendedby|           joindate|zipcode|     telephone|recommendedbyname|
+-----+---------------+--------------------+-------------+-------------------+-------+--------------+-----------------+
|    0|    GUEST GUEST|               GUEST|         NULL|2012-07-01 00:00:00|      0|(000) 000-0000|  NOT RECOMMENDED|
|    1|   Darren Smith|8 Bloomsbury Clos...|         NULL|2012-07-02 12:02:05|   4321|  555-555-5555|  NOT RECOMMENDED|
|    2|    Tracy Smith|8 Bloomsbury Clos...|         NULL|2012-07-02 12:08:23|   4321|  555-555-5555|  NOT RECOMMENDED|
|    3|     Tim Rownam|23 Highway Way, B...|         NULL|2012-07-03 09:32:15|  23423|(844) 693-0723|  NOT RECOMMENDED|
|    4|Janice Joplette|20 Crossing Road,...|            1|2012-07-03 10:25:05|    234|(833) 942-4710|     Darren Smith|
|    5| Gerald Butters|1065 Huntingdon A

In [15]:
# Save the dataframe to Minio
members_df.write\
    .format('csv')\
    .option("header", "true")\
    .mode("overwrite")\
    .save(f"s3a://{s3_bucket_name}/{transformed_prefix}/members")

26/01/09 17:02:49 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/01/09 17:02:49 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


### 2.2 Transforming the bookings

We need a new bookings table that shows

- `bookid` : booking id
- `memberid`: member id
- `member` : the full name of the member that made the booking
- `facilityid`: facility id
- `facility`: the name of the facility
- `starttime`: start time of the booking
- `endtime`: end time of the booking
- `slots`: amount of hours of the booking
- `cost`: the cost of the booking, the per hour cost by the total of hours


In [1]:
# Load all the dataframes you need from Minio

bookings_df = spark.read.csv(
    f"s3a://{s3_bucket_name}/{s3_landing_prefix}/bookings",
    header=True,      # Use first row as column names
    inferSchema=True  # Automatically detect data types
)
members_df = spark.read.csv(
    f"s3a://{s3_bucket_name}/{s3_landing_prefix}/members",
    header=True,      # Use first row as column names
    inferSchema=True  # Automatically detect data types
)
facilities_df = spark.read.csv(
    f"s3a://{s3_bucket_name}/{s3_landing_prefix}/facilities",
    header=True,      # Use first row as column names
    inferSchema=True  # Automatically detect data types
)
full_df = (bookings_df
    .join(members_df, bookings_df.memid == members_df.memid, "left_outer") 
    .join(facilities_df, bookings_df.facid == facilities_df.facid, "inner")
)



NameError: name 's3_bucket_name' is not defined

In [12]:
# Create the new dataframe
final_df = (full_df.withColumn("member", F.concat_ws(" ", F.col("firstname"), F.col("surname")))
           .withColumn("starttime", F.))
final_df.show()

+------+-----+-----+-------------------+-----+-----+-------+---------+--------------------+-------+--------------+-------------+-------------------+-----+---------------+----------+---------+-------------+------------------+------------+
|bookid|facid|memid|          starttime|slots|memid|surname|firstname|             address|zipcode|     telephone|recommendedby|           joindate|facid|           name|membercost|guestcost|initialoutlay|monthlymaintenance|      member|
+------+-----+-----+-------------------+-----+-----+-------+---------+--------------------+-------+--------------+-------------+-------------------+-----+---------------+----------+---------+-------------+------------------+------------+
|     0|    3|    1|2012-07-03 11:00:00|    2|    1|  Smith|   Darren|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2012-07-02 12:02:05|    3|   Table Tennis|       0.0|      5.0|          320|                10|Darren Smith|
|     1|    4|    1|2012-07-03 08:00:00|    2|  

In [ ]:
new_bookings_table = (final_df.select(
    F.col("bookid"),
    F.col("memid").alias("memberid"),
    F.col("member"),
    F.col("facid").alias("facilityid"),
    F.col("name").alias("facility"),
    F.col("starttime"),
    F.col("endtime"),
    F.col("slots"),
    F.col("cost")
))

full_df = (bookings_df
    .join(members_df, bookings_df.memid == members_df.memid, "left_outer") # Usamos left_outer para no perder invitados (memid=0)
    .join(facilities_df, bookings_df.facid == facilities_df.facid, "inner")
)

# Paso 3.2: Crear las nuevas columnas calculadas
final_df = (full_df
    # Crear la columna 'member' con el nombre completo. Si es un invitado (memid=0), el nombre será NULL.
    .withColumn("member", F.concat_ws(" ", F.col("firstname"), F.col("surname")))
    
    # ----> NUEVO: Calcular el endtime <----
    # Convertimos starttime a segundos (epoch), le sumamos la duración en segundos (slots * 3600), y volvemos a convertir a timestamp.
    .withColumn("endtime", F.from_unixtime(F.unix_timestamp("starttime") + F.col("slots") * 3600).cast("timestamp"))
    
    # ----> NUEVO: Calcular el coste de forma condicional <----
    .withColumn("cost",
        F.when(F.col("memid") == 0, 
               F.col("slots") * F.col("guestcost")  # Si es invitado
        ).otherwise(
               F.col("slots") * F.col("membercost")  # Si es socio
        )
    )
)


In [ ]:
# Save the result dataframe
new_bookings_table.write\
    .format('csv')\
    .option("header", "true")\
    .mode("overwrite")\
    .save(s3_spark_file_url)

## 3. Reporting 

You will have to generate reports with the following information:

- 3.1 `facility_report_monthly`: For each facility the total bookings, hours booked and revenuee (per month) ordered by facility
- 3.2 `facility_report_yearly`: For each facility the total bookings, hours booked and revenue
- 3.3 `member_report_monthly`: For each member the total boookings, total hours used, total money spent, and members recommended (per month)
- 3.4 `member_report_yearly`: For each member the total hours used and members recommended

Only consider the 2012 year

You can use any data that you want, save the reports with the prefix `lab2/reports`


In [ ]:
s3_reports_prefix = "lab2/reports"

Using the bookings report we built in the previous section we already have some of the work done

In [ ]:
# Load your transformed bookings dataframe here!

## 3.1 Facility monthly report

For breaking down the report into months we will need a rolling dates table, this a table where we will have all the possible dates in a given period of time. In this case we will cosider the entire 2012 year.

In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Generate all the possible dates (remember we only need a month) here ...

# dates_df = ...

## 3.2 Facility yearly report

This is quite simple, just an aggregation over the monthly report

## 3.3 Members monthly report

Like the facility one but with a little twist

## 3.4 Members yearly report

You should be able to do this very easily